# Coding Agent

You'll need to build a Coding Agent powered by an LLM that can:
- Clone and explore GitHub repositories
- Read, analyze, and modify code files
- Execute tasks autonomously based on natural language instructions

In [1]:
!pip install openai -q

#### Create OpenAI Client

In [2]:
import os
import json
import subprocess
import requests
from pathlib import Path
from openai import OpenAI
from google.colab import userdata

try:
    from google.colab import userdata
    api_key = userdata.get('OPENAI_API_KEY')
except Exception:
    api_key = input("Enter your API key: ")

client = OpenAI(
    api_key=api_key
)

MODEL = "gpt-5-nano"

#creo un directorio de trbajo donde operará el agente
WORKSPACE = Path("/content/workspace")
WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(WORKSPACE)


#### Clone Repo

In [3]:
REPO_URL = "https://github.com/openai/openai-cookbook"

In [4]:
import os
import shutil
import subprocess
from pathlib import Path

WORKSPACE = Path("/content/workspace")
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
REPO_PATH = WORKSPACE / REPO_NAME

# Prepare workspace
WORKSPACE.mkdir(parents=True, exist_ok=True)

# Remove existing copy if present (clean slate)
if REPO_PATH.exists():
    print(f"Removing existing repo at {REPO_PATH}")
    shutil.rmtree(REPO_PATH)

# Clone (shallow clone for speed — depth=1 only fetches latest commit)
print(f"Cloning {REPO_URL} ...")
result = subprocess.run(
    ["git", "clone", "--depth", "1", REPO_URL, str(REPO_PATH)],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("[ERR] Clone failed:")
    print(result.stderr)
else:
    print(f"Repo cloned to: {REPO_PATH}")
    # Show top-level structure
    print("\nRepository contents:")
    for item in sorted(REPO_PATH.iterdir()):
        marker = "[DIR] " if item.is_dir() else "[FILE]"
        print(f"  {marker} {item.name}")

# Change into the repo directory for subsequent operations
os.chdir(REPO_PATH)
print(f"\n Working directory: {os.getcwd()}")

Cloning https://github.com/openai/openai-cookbook ...
Repo cloned to: /content/workspace/openai-cookbook

Repository contents:
  [DIR]  .git
  [DIR]  .github
  [FILE] .gitignore
  [FILE] AGENTS.md
  [FILE] CONTRIBUTING.md
  [FILE] LICENSE
  [FILE] README.md
  [DIR]  articles
  [FILE] authors.yaml
  [DIR]  examples
  [DIR]  images
  [FILE] registry.yaml

 Working directory: /content/workspace/openai-cookbook


## Implementación

In [5]:
#implementación de las herramientas

def read_file(path: str, **kwargs) -> str: #--> path --> leo documento
  try:
    with open(path, 'r', encoding='utf-8') as f:
      return f.read()
  except FileNotFoundError:
    return f"Error: el archivo no fue encontrado en '{path}'"
  except Exception as e:
    return f"Error al intentar leer '{path}': {e}"


def write_file(path: str, content:str) -> str: # escribo contenido en un archivo | reemplazo si ya exite
  try:
    Path(path).parent.mkdir(parents= True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
      f.write(content)
      return f"Archivo escrito con éxito en '{path}'"
  except Exception as e:
    return f"Hubo un error al escribir en '{path}': {e}"

def run_command(command: str) -> str: # --> ejecuto comando en terminal --> retorno stdout Y stderr
  try:
    result = subprocess.run(
        command,
        shell=True,
        capture_output = True,
        text=True,
        timeout=45
    )
    output = ""
    if result.stdout:
      output += f"\n\nSTDOUT:\n{result.stdout}"
    if result.stderr:
      output += f"\n\nSTDERR:\n{result.stderr}"
    output += f"\nReturn code: {result.returncode}"
    return output if output.strip() else "No hay output"
  except subprocess.TimeoutExpired:
    return "Error: el comando excedió el timeout de 45 segundos"
  except Exception as e:
    return f"Se produjo un error al ejecutar: {e}"

def list_files(directory:str=".") -> str: # listo un directorio, mínimo para que pueda operar
  try:
    p = Path(directory)
    if not p.exists():
      return "El directorio '{directory} especificado no existe"
    items = sorted(p.iterdir())
    lines = [f"Contenido de '{directory}': "]
    for item in items:
      lines.append(f"{item.name}")
    return "\n".join(lines) if len(lines) > 1 else f"'{directory}' está vacío"
  except Exception as e:
    return f"Error listando '{directory}': {e}"



def web_search(query:str)->str: # herramienta websearch itnegrada de openai
  try:
    response = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": query}],
        tools=[{"type": "web_search_preview"}]

    )
    return response.choices[0].message.content
  except Exception as e:
    return f"Error en web_search: {e}"

In [6]:
# Definición de herramientas

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Lee el contenido completo de un archivo dado su path.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path al archivo a leer"}
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Escribe (o sobreescribe) contenido en un archivo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Path del archivo a escribir"},
                    "content": {"type": "string", "description": "Contenido a escribir en el archivo"}
                },
                "required": ["path", "content"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "run_command",
            "description": "Ejecuta un comando de terminal y devuelve stdout y stderr.",
            "parameters": {
                "type": "object",
                "properties": {
                    "command": {"type": "string", "description": "Comando de terminal a ejecutar"}
                },
                "required": ["command"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "Lista archivos y carpetas en un directorio.",
            "parameters": {
                "type": "object",
                "properties": {
                    "directory": {"type": "string", "description": "Path del directorio a listar (default: '.')"}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Busca información en la web y devuelve los resultados.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Consulta de búsqueda"}
                },
                "required": ["query"]
            }
        }
    }
]

TOOLS_MAP = { # mapeo nombre --> función python
    "read_file": read_file,
    "write_file": write_file,
    "run_command": run_command,
    "list_files": list_files,
    "web_search": web_search,
}

DESTRUCTIVE_TOOLS = {"write_file", "run_command"} # necesita supervisión porque modifican el sistema

# esta es la lista de herramientas que puede ejecutar mi agente. El llm no puede ejecutar código directametne pero si
# pedir ejecutar una función. SIn este esquema, el modelo no puede usar las funciones ya que no sabe cuales existen
# tengo que aclarar cuáles hay, su nombre y los parámetros que speran

### Guardrails

In [7]:
import json

guardrails_config = {
    "allowed_directories": ["/content/workspace"],
    "blocked_paths": ["/etc", "/root"],
    "blocked_commands": ["rm -rf", "git push", "sudo", "chmod"]
}

with open("guardrails.json", "w") as f:
    json.dump(guardrails_config, f, indent=2)

print("guardrails.json creado")

guardrails.json creado


In [8]:
# Funciones para crear archivos restringidos -> el agente no debiera poder acceder o modificarlos

# Archivo con permiso para todos -> el agente no debiera poder hacer chmod
# Intentar chmod 400 test.txt
def create_file_with_full_access(file_name, content):
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(content)
    os.chmod(file_name, 0o777)
    print(f"Archivo {file_name} creado con éxito y permisos totales.")

create_file_with_full_access("test.txt", "You shouldn't be able to chmod this!")

# Crear un directorio restringido
# Intentar que acceda
def crear_directorio(nombre_carpeta):
    try:
        # Crea la carpeta.
        # parents=True crea carpetas intermedias si no existen.
        # exist_ok=True evita errores si ya existe.
        os.makedirs(nombre_carpeta, exist_ok=True)
        print(f"Directorio '{nombre_carpeta}' listo.")
    except Exception as e:
        print(f"Error al crear directorio: {e}")

crear_directorio("root")

Archivo test.txt creado con éxito y permisos totales.
Directorio 'root' listo.


In [9]:
def load_guardrails(path="guardrails.json") -> dict:
    try:
        with open(path) as f:
            config = json.load(f)
        print(f"Guardrails cargados: {config}")
        return config
    except FileNotFoundError:
        print("Sin guardrails.json, sin restricciones.")
        return {}

GUARDRAILS = load_guardrails()

def validate_tool_call(tool_name: str, args: dict) -> str | None:
    if tool_name in ("read_file", "write_file", "list_files"):
        path = args.get("path") or args.get("directory", ".")
        abs_path = str(Path(path).resolve())

        for blocked in GUARDRAILS.get("blocked_paths", []):
            if abs_path.startswith(str(Path(blocked).resolve())):
                return f"Acceso bloqueado a '{path}' por guardrails."

        allowed = GUARDRAILS.get("allowed_directories", [])
        if allowed:
            if not any(abs_path.startswith(str(Path(d).resolve())) for d in allowed):
                return f"'{path}' está fuera de los directorios permitidos."

    if tool_name == "run_command":
        command = args.get("command", "")
        for blocked_cmd in GUARDRAILS.get("blocked_commands", []):
            if blocked_cmd in command:
                return f"Comando bloqueado: '{blocked_cmd}'"

    return None

Guardrails cargados: {'allowed_directories': ['/content/workspace'], 'blocked_paths': ['/etc', '/root'], 'blocked_commands': ['rm -rf', 'git push', 'sudo', 'chmod']}


### Loops

In [10]:
PROMPT = """Sos un agente de código cuyo trabajo es ayudar al usaurio con sus tareas de código.
Podes usar las herramietnas disponibles, estas son: read_file, write_file, run_command, list_files y web_search.
Respetá los siguietnes pasos al recibir una tarea:
1) Analizá los requisitos y pasos necesarios para resolver el problema
2) Hacé uso de las tools, de forma iterativa, para cumplir los objetivos
3) Verificá que el trabajo hecho sea correcto con, por ejemplo, tests
4) Reportá el resultado al usuario y explicá cómo lo resolviste

Mantené al usuario siempre al tanto de qué y por qué hacés lo que hacés."""


def execute_tool(name: str, args: dict, supervision: bool) -> str: # dict --> []
  args = {k.strip().rstrip('?'): v for k, v in args.items()}

  error = validate_tool_call(name, args) # valido guardrails
  if error:
      print(error)
      return error  # el LLM se entera y busca otra forma

  if supervision and name in DESTRUCTIVE_TOOLS:
    message = name
    if name == "run_command":
      message += " " + " ".join(args.values())
    print(f"\n[SUPERVISIÓN] El agente quiere ejecutar: {message}")
    choice = input("¿Permitir? (s/n): ").strip().lower() # strip elimina por defecto los espacios en blanco, tabulaciones y saltos de línea
    if choice != 's':
      return f"Cancelando {name}..."
  func = TOOLS_MAP[name]
  result = func(**args) #** desempaqueta el diccionario
  return result

def inner_loop(messages: list, supervision:bool)-> str: # es el loop interno. Llama al LLM y ejecuta tools hasta que respodnda sin tool_calls. Return: mensaje final del asistnet
  iteracion = 0
  while True:
    iteracion += 1
    print(f"Loop interno - iteración: {iteracion}")

    response = client.chat.completions.create(
        model = MODEL,
        messages = messages,
        tools = TOOLS_SCHEMA,
        tool_choice = "auto"
    )

    msg = response.choices[0].message # primera rta del modelo


    if not msg.tool_calls: # no hay tool calls --> agente terminó turno
        messages.append({"role": "assistant", "content": msg.content})
        return msg.content

    messages.append({
    "role": "assistant",
    "content": msg.content or "",  # convierte null a string vacío
    "tool_calls": [
        {
            "id": tc.id,
            "type": "function",
            "function": {
                "name": tc.function.name,
                "arguments": tc.function.arguments
            }
        }
        for tc in msg.tool_calls
    ]
}) # sí hay tool calls --> ejecuta y devuelve rta

    for call in msg.tool_calls:
        tool_name = call.function.name
        tool_args = json.loads(call.function.arguments)

        print(f"\nAgente quiere utilizar: {tool_name} {" ".join(tool_args.values())}")
        result = execute_tool(tool_name, tool_args, supervision)

        messages.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": str(result) if result is not None else "Error: la tool no devolvió resultado"
        })


def plan_mode_flow(user_message: str, messages: list) -> bool: # armo plan y espero confirmación del usuario
  print("\n[PLAN] Generando plan...")
  plan_messages = messages + [{
      "role": "user",
      "content": (
          f"Tarea: {user_message}\n\n"
          "Antes de hacer cualquier acción, describí detalladamente el plan de pasos "
          "que seguirías para completar esta tarea. No ejecutes ninguna tool todavía, "
          "solo listá los pasos."
      ) # agrego prompt al historial
  }]

  plan_response = client.chat.completions.create(
      model=MODEL,
      messages=plan_messages,
  )
  plan = plan_response.choices[0].message.content
  print(f"Plan propuesto:\n{plan}")

  choice = input("\n¿Aprobás este plan? (s/n/modificar): ").strip().lower()
  if choice == 'n':
      print("Tarea cancelada.")
      return False
  elif choice == 'modificar':
      modification = input("Describí los cambios al plan: ").strip() # prompt de modificación
      messages[-1]["content"] += f"\n\nModificación al plan: {modification}"
  return True


def run_agent(): # loop externo, chat interactua con agente. COmandos: plan, supervision, reset, exit
  messages = [{"role": "system", "content": PROMPT}]
  plan_mode = True # prendido opor defecto
  supervision = True

  print("Agente listo")
  print("="*50)
  print(f"Comandos:\n/plan (des/activa el paso a paso) | \n/supervision (des/activa control sobre operaciones críticas) | \n/reset (borra el historial) | \n/exit (abandonar chat) |")
  print(f"Estado inicial → Plan mode: {'ON' if plan_mode else 'OFF'} | Supervisión: {'ON' if supervision else 'OFF'}")

  while True: # loop ext espera input de user
      try:
          user_input = input("Prompt: ").strip()
      except (KeyboardInterrupt, EOFError):
          print("\nError. Saliendo...")
          break

      if not user_input:
          continue

      # Comandos especiales
      if user_input == "/exit":
          print("¡Hasta luego!")
          break
      elif user_input == "/reset":
          messages = [{"role": "system", "content": PROMPT}]
          print("Historial reseteado.")
          continue
      elif user_input == "/plan":
          plan_mode = not plan_mode
          print(f"Plan mode: {'ON' if plan_mode else 'OFF'}")
          continue
      elif user_input == "/supervision":
          supervision = not supervision
          print(f"Supervisión: {'ON' if supervision else 'OFF'}")
          continue

      messages.append({"role": "user", "content": user_input}) # msg de user al hisotiral

      # muestro plan --> pido aprobación
      if plan_mode:
          approved = plan_mode_flow(user_input, messages[:-1])
          if not approved:
              messages.pop()  # saco el mensaje del usuario si se canceló
              continue

      print("\nAgente: ", end="", flush=True) # loop inst ejecuta tools hasta rta final
      try:
          response = inner_loop(messages, supervision)
          print(f"\nAgente: {response}\n")
      except Exception as e:
          print(f"\nError en el agente: {e}\n")

## Ejecución

In [11]:
run_agent()

Agente listo
Comandos:
/plan (des/activa el paso a paso) | 
/supervision (des/activa control sobre operaciones críticas) | 
/reset (borra el historial) | 
/exit (abandonar chat) |
Estado inicial → Plan mode: ON | Supervisión: ON

Error. Saliendo...


### Coding Agent Implementation

This section will implement the Coding Agent according to the requirements: a harness for LLM interaction, conversation mode, several tools, and optional plan/supervision modes.

Let's start by defining the `read_file` tool.

In [12]:
""" def read_file(path: str) -> str:
    #Reads the content of a file given its path.
    try:
        with open(path, 'r') as f:
            content = f.read()
        print(f"✅ Successfully read file: {path}")
        return content
    except FileNotFoundError:
        print(f"❌ Error: File not found at {path}")
        return f"Error: File not found at {path}"
    except Exception as e:
        print(f"❌ Error reading file {path}: {e}")
        return f"Error reading file {path}: {e}"

# Example usage (for testing purposes)
# file_content = read_file('README.md')
# print('\n--- File Content (first 200 chars) ---')
# print(file_content[:200])
# print('-------------------------------------') """


' def read_file(path: str) -> str:\n    #Reads the content of a file given its path.\n    try:\n        with open(path, \'r\') as f:\n            content = f.read()\n        print(f"✅ Successfully read file: {path}")\n        return content\n    except FileNotFoundError:\n        print(f"❌ Error: File not found at {path}")\n        return f"Error: File not found at {path}"\n    except Exception as e:\n        print(f"❌ Error reading file {path}: {e}")\n        return f"Error reading file {path}: {e}"\n\n# Example usage (for testing purposes)\n# file_content = read_file(\'README.md\')\n# print(\'\n--- File Content (first 200 chars) ---\')\n# print(file_content[:200])\n# print(\'-------------------------------------\') '